<a href="https://colab.research.google.com/github/2303A51528/reinforcement_learning_1528/blob/main/practical_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Let's define a simple Grid-World environment representing 'grid_env_det.py'
class GridWorldEnv:
    def __init__(self):
        # States: S0, S1, S2 (terminal), S3, S4, S5 (terminal)
        # Grid layout:
        # S0  S1  S2(T)
        # S3  S4  S5(T)
        self.states = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5']
        self.terminal_states = ['S2', 'S5']
        self.actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']

        # Deterministic transitions: (state, action) -> (next_state, reward)
        self.transitions = {
            ('S0', 'RIGHT'): ('S1', -1),
            ('S0', 'DOWN'): ('S3', -1),
            ('S1', 'LEFT'): ('S0', -1),
            ('S1', 'RIGHT'): ('S2', 10), # Reaching terminal S2
            ('S1', 'DOWN'): ('S4', -1),
            ('S3', 'UP'): ('S0', -1),
            ('S3', 'RIGHT'): ('S4', -1),
            ('S4', 'UP'): ('S1', -1),
            ('S4', 'LEFT'): ('S3', -1),
            ('S4', 'RIGHT'): ('S5', 5), # Reaching terminal S5
        }

    def step(self, state, action):
        if state in self.terminal_states:
            return state, 0, True

        key = (state, action.upper())
        if key in self.transitions:
            next_state, reward = self.transitions[key]
        else:
            # If action leads out of grid, agent stays in the same state with a penalty
            next_state = state
            reward = -1

        is_terminal = next_state in self.terminal_states
        return next_state, reward, is_terminal

env = GridWorldEnv()

In [ ]:
# Policy (П) = [S0: Right, S1: Down, S3: Right, S4: UP]
policy = {
    'S0': 'RIGHT',
    'S1': 'DOWN',
    'S3': 'RIGHT',
    'S4': 'UP'
}

In [5]:
policy = {
    'S0': 'RIGHT',
    'S1': 'DOWN',
    'S3': 'RIGHT',
    'S4': 'UP'
}

def get_trajectory_from_state(env, policy, start_state, max_steps=100):
    trajectory = []
    current_state = start_state
    for _ in range(max_steps):
        if current_state in env.terminal_states:
            break
        action = policy.get(current_state)
        if not action:
            break
        next_state, reward, done = env.step(current_state, action)
        trajectory.append((current_state, action, reward))
        current_state = next_state
        if done:
            break
    return trajectory

def state_value_function(env, policy, start_state, gamma=0.9):
    trajectory = get_trajectory_from_state(env, policy, start_state)
    g = 0
    for i, (_, _, reward) in enumerate(trajectory):
        g += (gamma ** i) * reward
    return g

# Calculate and display the state value for S0
s0_value = state_value_function(env, policy, 'S0')
print(f"State Value for state 'S0': {s0_value}")

State Value for state 'S0': -9.999734386011122


In [6]:
policy = {
    'S0': 'RIGHT',
    'S1': 'DOWN',
    'S3': 'RIGHT',
    'S4': 'UP'
}

def action_value_function(env, policy, start_state, first_action, gamma=0.9):
    # First step taken explicitly
    next_state, reward, done = env.step(start_state, first_action)
    g = reward

    # Remaining trajectory following the policy
    if not done:
        remaining_trajectory = get_trajectory_from_state(env, policy, next_state)
        for i, (_, _, r) in enumerate(remaining_trajectory):
            g += (gamma ** (i + 1)) * r
    return g

# Calculate and display action value for S0 and action 'UP'
s0_up_value = action_value_function(env, policy, 'S0', 'UP')
print(f"Action Value for state 'S0' and action 'UP': {s0_up_value}")

Action Value for state 'S0' and action 'UP': -9.99976094741001
